# DeepSeek V4 Standard vs Fast

This notebook sends the same prompts to the current DeepSeek V4 profiles used by the app:

- **DeepSeek Fast**: `deepseek-v4-flash`, profile id `deepseek-chat`, thinking disabled
- **DeepSeek Standard**: `deepseek-v4-pro`, profile id `deepseek-reasoner`, thinking enabled

It loads `.env.local` and `.env`, calls the OpenAI-compatible `/chat/completions` endpoint, displays each return value, and saves a JSON report under `artifacts/deepseek_v4_compare_notebook/`.


In [ ]:
from __future__ import annotations

from datetime import datetime
from html import escape
from pathlib import Path
import json
import os
import time

import requests
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "package.json").exists() and (candidate / "test_ipynb").exists():
            return candidate
    return Path(r"f:\Documents\GitHub\AI_TRPG_616\version 3.0")


repo_root = find_repo_root(Path.cwd())
load_dotenv(repo_root / ".env.local", override=False)
load_dotenv(repo_root / ".env", override=False)

output_dir = repo_root / "artifacts" / "deepseek_v4_compare_notebook"
output_dir.mkdir(parents=True, exist_ok=True)

BASE_URL = (os.getenv("TRPG_DEEPSEEK_BASE_URL") or "https://api.deepseek.com").rstrip("/")
API_KEY = os.getenv("DEEPSEEK_API_KEY") or os.getenv("TRPG_SERVER_PROXY_API_KEY")

if not API_KEY:
    raise RuntimeError("Missing DEEPSEEK_API_KEY or TRPG_SERVER_PROXY_API_KEY in .env/.env.local/environment.")

print(f"Repo root: {repo_root}")
print(f"Output dir: {output_dir}")
print(f"Base URL: {BASE_URL}")


In [ ]:
def first_env(*keys: str, default: str | None = None) -> str | None:
    for key in keys:
        value = os.getenv(key)
        if value and value.strip():
            return value.strip()
    return default


def int_env(*keys: str, default: int) -> int:
    value = first_env(*keys)
    if not value:
        return default
    try:
        return int(value)
    except ValueError:
        return default


DEFAULT_TEMPERATURE = float(os.getenv("TRPG_SERVER_PROXY_TEMPERATURE", "0.7"))
DEFAULT_MAX_TOKENS = int(os.getenv("TRPG_SERVER_PROXY_MAX_TOKENS", "1200"))

PROFILES = [
    {
        "profile_id": "deepseek-chat",
        "label": "DeepSeek V4 Fast",
        "model": first_env(
            "TRPG_DEEPSEEK_CHAT_MODEL",
            "TRPG_DEEPSEEK_MODEL",
            "TRPG_SERVER_PROXY_MODEL",
            default="deepseek-v4-flash",
        ),
        "timeout_sec": int_env("TRPG_SERVER_PROXY_TIMEOUT_MS", default=60_000) / 1000,
        "extra_payload": {
            "temperature": DEFAULT_TEMPERATURE,
            "thinking": {"type": "disabled"},
        },
    },
    {
        "profile_id": "deepseek-reasoner",
        "label": "DeepSeek V4 Standard",
        "model": first_env(
            "TRPG_DEEPSEEK_STANDARD_MODEL",
            "TRPG_DEEPSEEK_REASONER_MODEL",
            default="deepseek-v4-pro",
        ),
        "timeout_sec": int_env(
            "TRPG_DEEPSEEK_STANDARD_TIMEOUT_MS",
            "TRPG_DEEPSEEK_REASONER_TIMEOUT_MS",
            "TRPG_SERVER_PROXY_TIMEOUT_MS",
            default=420_000,
        ) / 1000,
        "extra_payload": {
            "thinking": {"type": "enabled"},
            "reasoning_effort": "high",
        },
    },
]

# Set this to {"deepseek-chat"} or {"deepseek-reasoner"} for a single-profile run.
ONLY_PROFILES: set[str] | None = None

active_profiles = [p for p in PROFILES if ONLY_PROFILES is None or p["profile_id"] in ONLY_PROFILES]
for profile in active_profiles:
    print(f"{profile['label']}: profile={profile['profile_id']}, model={profile['model']}, timeout={profile['timeout_sec']}s")


In [ ]:
TEST_CASES = [
    {
        "id": "trpg_narration",
        "label": "TRPG narrator continuation",
        "system_prompt": "你是中文 TRPG 主持人。延续场景，给出具体、可继续的结果，不要替玩家做决定。",
        "user_prompt": """
故事标题：VHS复古恐怖录像带

最近上下文：
- 玩家在废弃录像出租店的后仓找到一台仍在运转的录像机。
- 电视屏幕里出现了玩家刚才经过的街角，但画面时间似乎慢了三分钟。
- 柜台抽屉里有一张会员卡，背面写着：请不要倒带。

本轮玩家行动：我把会员卡放到录像机旁边，盯着电视里的街角，等待那三分钟后的事情发生。

任务：写出主持人的下一段推进，控制在 180 到 320 个中文字符。
""".strip(),
    },
    {
        "id": "rag_rule_decision",
        "label": "RAG-style rule decision",
        "system_prompt": "你是 TRPG 规则裁判。只根据给定规则片段判断，不要编造额外规则。",
        "user_prompt": """
规则片段：
1. 当玩家做出有风险且结果不确定的行动时，主持人要求一次判定。
2. 普通难度判定目标值为 12，困难难度为 16。
3. 若行动使用了明确有利的工具，判定结果 +2。
4. 若失败但只差 1 到 2 点，主持人应给出代价成功或部分成功。

场景：玩家想用手电筒和会员卡反光观察录像机内部是否有卡住的磁带。主持人认为这是普通难度。玩家掷骰结果总和为 10。

任务：判断是否成功，并说明主持人下一步应该怎么处理。回答要简洁。
""".strip(),
    },
    {
        "id": "json_asset_plan",
        "label": "JSON asset plan",
        "system_prompt": "你是 TRPG 内容资产规划助手。必须返回合法 JSON object，不要输出 Markdown。",
        "user_prompt": """
Return a valid json object for this asset plan. The top-level keys must be:
- cover_prompt: string
- locations: array of short strings
- npc_portraits: array of short strings

剧本：VHS复古恐怖录像带。核心地点是雨夜录像出租店、后仓、地下剪辑室。主要 NPC 是沉默的店员和只存在于录像里的失踪顾客。
""".strip(),
        "response_format": {"type": "json_object"},
    },
]

print(f"Loaded {len(TEST_CASES)} test cases.")
for case in TEST_CASES:
    print(f"- {case['id']}: {case['label']}")


In [ ]:
def normalize_text(value) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        chunks = []
        for item in value:
            if isinstance(item, str):
                chunks.append(item)
            elif isinstance(item, dict) and isinstance(item.get("text"), str):
                chunks.append(item["text"])
        return "".join(chunks).strip()
    return ""


def summarize_usage(usage: dict) -> str:
    if not usage:
        return "n/a"
    prompt_tokens = usage.get("prompt_tokens") or usage.get("input_tokens") or 0
    completion_tokens = usage.get("completion_tokens") or usage.get("output_tokens") or 0
    total_tokens = usage.get("total_tokens") or (prompt_tokens + completion_tokens)
    cached_tokens = (
        usage.get("prompt_cache_hit_tokens")
        or (usage.get("prompt_tokens_details") or {}).get("cached_tokens")
        or (usage.get("input_tokens_details") or {}).get("cached_tokens")
        or 0
    )
    return f"prompt={prompt_tokens}, completion={completion_tokens}, total={total_tokens}, cached={cached_tokens}"


def build_messages(system_prompt: str, user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


def call_deepseek(profile: dict, case: dict) -> dict:
    url = f"{BASE_URL}/chat/completions"
    payload = {
        "model": profile["model"],
        "messages": build_messages(case["system_prompt"], case["user_prompt"]),
        "max_tokens": DEFAULT_MAX_TOKENS,
        "stream": False,
        **profile["extra_payload"],
    }
    if case.get("response_format"):
        payload["response_format"] = case["response_format"]

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }

    started_at = time.perf_counter()
    response = requests.post(url, headers=headers, json=payload, timeout=profile["timeout_sec"])
    duration_sec = round(time.perf_counter() - started_at, 3)

    record = {
        "case_id": case["id"],
        "case_label": case["label"],
        "profile_id": profile["profile_id"],
        "profile_label": profile["label"],
        "model": profile["model"],
        "duration_sec": duration_sec,
        "status_code": response.status_code,
        "ok": response.ok,
    }

    try:
        data = response.json()
    except ValueError:
        data = {"raw_text": response.text}

    record["raw_response"] = data

    if not response.ok:
        record["error"] = response.text[:4000]
        return record

    choice = (data.get("choices") or [{}])[0]
    message = choice.get("message") or {}
    record.update(
        {
            "finish_reason": choice.get("finish_reason"),
            "content": normalize_text(message.get("content")),
            "reasoning_content": normalize_text(message.get("reasoning_content")),
            "usage": data.get("usage") or {},
        }
    )
    return record


In [ ]:
results = []

for case in TEST_CASES:
    print(f"\n=== {case['label']} ===")
    for profile in active_profiles:
        print(f"Calling {profile['label']} ({profile['model']}) ...", flush=True)
        result = call_deepseek(profile, case)
        results.append(result)
        if result["ok"]:
            print(
                f"  ok in {result['duration_sec']}s, finish={result.get('finish_reason')}, "
                f"usage={summarize_usage(result.get('usage') or {})}"
            )
        else:
            print(f"  failed HTTP {result['status_code']} in {result['duration_sec']}s")

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
report = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "base_url": BASE_URL,
    "profiles": active_profiles,
    "test_cases": TEST_CASES,
    "results": results,
}
save_path = output_dir / f"deepseek_v4_standard_fast_{timestamp}.json"
save_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\nSaved report: {save_path}")


In [ ]:
def build_summary_table(rows: list[dict]) -> str:
    headers = ["Case", "Profile", "Model", "HTTP", "Duration", "Finish", "Usage", "Reasoning"]
    html = ["<table>", "<thead><tr>"]
    for header in headers:
        html.append(f"<th style='text-align:left;padding:6px 10px;border-bottom:1px solid #ccc;'>{escape(header)}</th>")
    html.append("</tr></thead><tbody>")
    for row in rows:
        values = [
            row.get("case_label", ""),
            row.get("profile_label", ""),
            row.get("model", ""),
            str(row.get("status_code", "")),
            f"{row.get('duration_sec', '')}s",
            str(row.get("finish_reason") or "n/a"),
            summarize_usage(row.get("usage") or {}),
            "yes" if row.get("reasoning_content") else "no",
        ]
        html.append("<tr>")
        for value in values:
            html.append(f"<td style='vertical-align:top;padding:6px 10px;border-bottom:1px solid #eee;'>{escape(value)}</td>")
        html.append("</tr>")
    html.append("</tbody></table>")
    return "".join(html)


display(HTML(build_summary_table(results)))


In [ ]:
for row in results:
    title = f"## {row['case_label']} / {row['profile_label']} / `{row['model']}`"
    if not row["ok"]:
        display(Markdown(f"{title}\n\n**ERROR** HTTP {row['status_code']}\n\n```text\n{row.get('error', '')}\n```"))
        continue

    content = row.get("content") or ""
    reasoning = row.get("reasoning_content") or ""
    blocks = [
        title,
        f"- Duration: `{row['duration_sec']}s`",
        f"- Finish reason: `{row.get('finish_reason')}`",
        f"- Usage: `{summarize_usage(row.get('usage') or {})}`",
        "",
        "### Content",
        content,
    ]
    if reasoning:
        blocks.extend(["", "### Reasoning Content", reasoning])
    display(Markdown("\n".join(blocks)))


In [ ]:
# Optional: validate the JSON-mode case parsed as JSON.
json_rows = [row for row in results if row.get("case_id") == "json_asset_plan" and row.get("ok")]
for row in json_rows:
    print(f"\n{row['profile_label']} JSON parse check:")
    try:
        parsed = json.loads(row.get("content") or "")
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    except json.JSONDecodeError as exc:
        print(f"JSON parse failed: {exc}")
        print(row.get("content") or "")
